# Data integration

In [ ]:
# Python packages
#!pip3 install scanpy
#!pip3 install scvi-tools
#!pip3 install bbknn
#!pip3 install scib
#!pip3 install anndata2ri

import scanpy as sc
import scvi
#import bbknn
#import scib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import os

# R interface
from rpy2.robjects import pandas2ri
from rpy2.robjects import r
import rpy2.rinterface_lib.callbacks
import anndata2ri

pandas2ri.activate()
anndata2ri.activate()

%load_ext rpy2.ipython
#print(scvi.__version__)

In [ ]:
os.listdir()

In [ ]:

#working env
import os
os.chdir(r'\\isdsyn1-1.isd.med.uni-muenchen.de\BD-Bernhagen\Simon Ebert\snRNAseq_Python_env_normalMT')
os.listdir()

## Dataset

The dataset we will use to demonstrate data integration contains several samples of bone marrow mononuclear cells. These samples were originally created for the Open Problems in Single-Cell Analysis [NeurIPS Competition 2021](https://openproblems.bio/neurips_2021/) {cite}`Luecken2022-kv,Lance2022-yy`. The [10x Multiome](https://www.10xgenomics.com/products/single-cell-multiome-atac-plus-gene-expression) protocol was used which measures both RNA expression (scRNA-seq) and chromatin accessibility (scATAC-seq) in the same cells. The version of the data we use here was already pre-processed to remove low-quality cells.

Let's read in the dataset using **scanpy** to get an `AnnData` object.

In [ ]:
adata_raw = sc.read_h5ad(
    "combine_data_afer_QC.h5ad"
)
adata_raw.layers["logcounts"] = adata_raw.X
adata_raw

In [ ]:
np.max(adata_raw.X)

In [ ]:
label_key = "cell_type"
batch_key = "batch"

In [ ]:
adata_raw.obs[batch_key].value_counts()

In [ ]:
keep_batches = ["HFD", "HFD_L1"]
adata = adata_raw[adata_raw.obs[batch_key].isin(keep_batches)].copy()
adata

In [ ]:
adata.var["feature_types"].value_counts()

In [ ]:
adata = adata[:, adata.var["feature_types"] == "Gene Expression"].copy()
sc.pp.filter_genes(adata, min_cells=1)
adata

In [ ]:
adata.X = adata.layers["soupX_counts"].copy()
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
adata.layers["logcounts"] = adata.X.copy()

In [ ]:
np.max(adata.X)

In [ ]:
sc.pp.highly_variable_genes(adata)
sc.tl.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
adata

In [ ]:
adata.uns[batch_key +   "_colors"] = [
    "#1b9e77",
    "#d95f02",

]  # Set custom colours for batches
sc.pl.umap(adata, color=[label_key, batch_key], wspace=1)

In [ ]:
adata.write_h5ad("unintegrated_data_new.h5ad")

In [ ]:
from matplotlib.pyplot import rc_context

In [ ]:
adatatemp=adata.copy()

In [ ]:
marker_genes_Wirka2019 = {
"SMCs1":["Acta2","Tagln","Tnnt2","Tpm2","Myl9","Cnn1","Tpm1","Dstn","Sost","Lmod1","Myh11","Csrp2","Csrp1","Actg2","Ppp1r14a","Tcap","Hspb1","Cald1","Hspb6","Mylk"],
"SMCs2":["Acta2","Cnn1","Myl9","Myh11","Tagln","Tpm2","Sost","Flna","Itga8","Ppp1r14a","Dstn","Csrp1","Col18a1","Csrp2","Sh3bgr","Mylk","Myl6","Actn1","Tpm1","Nov"],
"Fibroblasts1":["Pi16","Clec3b","Gpx3","Serping1","Cygb","Dpep1","Smoc2","Gsn","C3","Igfbp4","Pcolce","Fbln1","Serpinf1","Entpd2","Ly6a","Abca8a","Cxcl1","Mfap5","Col3a1","Dpt"],
"Fibroblasts2":["Comp","Chad","Wif1","1500015O10Rik","Thbs1","Fmod","Angptl7","Prg4","Fxyd6","Prelp","Dkk3","Ccdc80","Meox1","Dcn","Ptn","Npy","Aldh1a1","Dpt","Itih5","Abi3bp"],
"Fibromyocytes":["Spp1","Ibsp","Fn1","Col2a1","Lcn2","Lum","Timp1","Tnfrsf11b","Mgp","1500015O10Rik","Clu","Frzb","Serpina3n","Serpine2","Sncg","Vcam1","Rarres1","Ltbp2","Lgals3","Crispld1","Crlf1","Anxa8","Bgn","Acan","Tnc","Scara3","Ank","C3","Sdc1","Cx3cl1"],
"Macrophages":["Cd74","Lyz2","C1qa","C1qb","H2-Aa","H2-Ab1","C1qc","H2-Eb1","Fcer1g","Wfdc17","Ctss","Cxcl2","Ccl8","Tyrobp","Pf4","Ccl6","Cd14","Ccl9","Il1b","Ctsc"],
"ECs1":["Ctla2a","Pecam1","Ly6c1","Cldn5","Gpihbp1","Egfl7","Cd36","Esam","Emcn","Icam2","Vwf","Cdh5","Tspan13","Eng","Slc9a3r2","Sdpr","Ptprb","Ecscr","Ramp2","Flt1"],
"ECs2":["Ccl21a","Nts","Lrg1","Cldn5","Mmrn1","Ecscr","Lbp","Clca3a1","Apold1","Egfl7","Pard6g","Ctla2a","Galnt18","Flt4","Emcn","Gpm6a","Gngt2","Pvrl2","Prox1","Icam2"],
"Pericytes":["Colec11","Ndufa4l2","Vtn","Higd1b","Ifitm1","Notch3","Nrip2","Olfr558","Ephx3","Casq2","Myo1b","Adamts4","Rgs5","Ngf","Adap2os","S1pr3","Rgs16","Igfbp5","Enpep","Parm1"],
"Epithelial cells":["Upk3b","Msln","Fxyd3","Nkain4","Krt19","Gpm6a","Krt7","Myl7","Upk1b","Cldn15","Rspo1","C2","Krt8","Wt1","Tmem151a","Dmkn","Lrrn4","Chst4","Dpp4","Slc39a8"],
"Neurons":["Plp1","Kcna1","Gfra3","Egfl8","Sostdc1","Mbp","Gpr37l1","Cadm4","Nrn1","Lgi4","Chl1","Tspan15","Fabp7","Adam23","Gfap","Cdh19","Plekhb1","Cmtm5","Mpz","Ttyh1"]
}

In [ ]:
def sort_and_score(adatatemp, marker_genes):
    
    marker_genes_in_data = {}
    for ct, markers in marker_genes.items():
        markers_found = []
        for marker in markers:
            if marker in adatatemp.var.index:
                markers_found.append(marker)
        if markers_found:
            marker_genes_in_data[ct] = markers_found

    for key in marker_genes_in_data.keys():
        sc.tl.score_genes(adatatemp, marker_genes_in_data[key], score_name=key)
        
    return adatatemp, marker_genes_in_data

In [ ]:
adatatemp, marker_genes_in_data = sort_and_score(adatatemp, marker_genes_Wirka2019)

In [ ]:
with rc_context({'figure.figsize': (6,6)}):
    sc.pl.umap(
    adatatemp,
    color=marker_genes_in_data.keys(), 
    frameon= False, ncols=3,
    cmap='RdYlBu_r', vmin=0, vmax=0.9, 
    size=10, sort_order=True
    )

# Use Harmony

In [ ]:
data_integrated_harmony=adata.copy()

In [ ]:
import scanpy.external as sce

In [ ]:
sce.pp.harmony_integrate(data_integrated_harmony, "batch")

In [ ]:
data_integrated_harmony.obsm['X_pca'] = data_integrated_harmony.obsm['X_pca_harmony']
sc.pp.neighbors(data_integrated_harmony)
sc.tl.umap(data_integrated_harmony)
sc.tl.leiden(data_integrated_harmony, resolution=0.1)

In [ ]:
sc.pl.umap(data_integrated_harmony, color=[label_key, batch_key], wspace=1)

In [ ]:
data_integrated_harmony.write_h5ad("integrated_harmony.h5ad")

In [ ]:
from matplotlib.pyplot import rc_context

In [ ]:
adatatemp=data_integrated_harmony.copy()

In [ ]:
marker_genes_Wirka2019 = {
"SMCs1":["Acta2","Tagln","Tnnt2","Tpm2","Myl9","Cnn1","Tpm1","Dstn","Sost","Lmod1","Myh11","Csrp2","Csrp1","Actg2","Ppp1r14a","Tcap","Hspb1","Cald1","Hspb6","Mylk"],
"SMCs2":["Acta2","Cnn1","Myl9","Myh11","Tagln","Tpm2","Sost","Flna","Itga8","Ppp1r14a","Dstn","Csrp1","Col18a1","Csrp2","Sh3bgr","Mylk","Myl6","Actn1","Tpm1","Nov"],
"Fibroblasts1":["Pi16","Clec3b","Gpx3","Serping1","Cygb","Dpep1","Smoc2","Gsn","C3","Igfbp4","Pcolce","Fbln1","Serpinf1","Entpd2","Ly6a","Abca8a","Cxcl1","Mfap5","Col3a1","Dpt"],
"Fibroblasts2":["Comp","Chad","Wif1","1500015O10Rik","Thbs1","Fmod","Angptl7","Prg4","Fxyd6","Prelp","Dkk3","Ccdc80","Meox1","Dcn","Ptn","Npy","Aldh1a1","Dpt","Itih5","Abi3bp"],
"Fibromyocytes":["Spp1","Ibsp","Fn1","Col2a1","Lcn2","Lum","Timp1","Tnfrsf11b","Mgp","1500015O10Rik","Clu","Frzb","Serpina3n","Serpine2","Sncg","Vcam1","Rarres1","Ltbp2","Lgals3","Crispld1","Crlf1","Anxa8","Bgn","Acan","Tnc","Scara3","Ank","C3","Sdc1","Cx3cl1"],
"Macrophages":["Cd74","Lyz2","C1qa","C1qb","H2-Aa","H2-Ab1","C1qc","H2-Eb1","Fcer1g","Wfdc17","Ctss","Cxcl2","Ccl8","Tyrobp","Pf4","Ccl6","Cd14","Ccl9","Il1b","Ctsc"],
"ECs1":["Ctla2a","Pecam1","Ly6c1","Cldn5","Gpihbp1","Egfl7","Cd36","Esam","Emcn","Icam2","Vwf","Cdh5","Tspan13","Eng","Slc9a3r2","Sdpr","Ptprb","Ecscr","Ramp2","Flt1"],
"ECs2":["Ccl21a","Nts","Lrg1","Cldn5","Mmrn1","Ecscr","Lbp","Clca3a1","Apold1","Egfl7","Pard6g","Ctla2a","Galnt18","Flt4","Emcn","Gpm6a","Gngt2","Pvrl2","Prox1","Icam2"],
"Pericytes":["Colec11","Ndufa4l2","Vtn","Higd1b","Ifitm1","Notch3","Nrip2","Olfr558","Ephx3","Casq2","Myo1b","Adamts4","Rgs5","Ngf","Adap2os","S1pr3","Rgs16","Igfbp5","Enpep","Parm1"],
"Epithelial cells":["Upk3b","Msln","Fxyd3","Nkain4","Krt19","Gpm6a","Krt7","Myl7","Upk1b","Cldn15","Rspo1","C2","Krt8","Wt1","Tmem151a","Dmkn","Lrrn4","Chst4","Dpp4","Slc39a8"],
"Neurons":["Plp1","Kcna1","Gfra3","Egfl8","Sostdc1","Mbp","Gpr37l1","Cadm4","Nrn1","Lgi4","Chl1","Tspan15","Fabp7","Adam23","Gfap","Cdh19","Plekhb1","Cmtm5","Mpz","Ttyh1"]
}

In [ ]:
def sort_and_score(adatatemp, marker_genes):
    
    marker_genes_in_data = {}
    for ct, markers in marker_genes.items():
        markers_found = []
        for marker in markers:
            if marker in adatatemp.var.index:
                markers_found.append(marker)
        if markers_found:
            marker_genes_in_data[ct] = markers_found

    for key in marker_genes_in_data.keys():
        sc.tl.score_genes(adatatemp, marker_genes_in_data[key], score_name=key)
        
    return adatatemp, marker_genes_in_data

In [ ]:
adatatemp, marker_genes_in_data = sort_and_score(adatatemp, marker_genes_Wirka2019)

In [ ]:
with rc_context({'figure.figsize': (6,6)}):
    sc.pl.umap(
    adatatemp,
    color=marker_genes_in_data.keys(), 
    frameon= False, ncols=3,
    cmap='RdYlBu_r', vmin=0, vmax=0.9, 
    size=10, sort_order=True
    )